# 01 — RAG ingestion

Jednorazowy pipeline do dodawania nowego dokumentu miejskiego do bazy Pismaka.

**Ważne:** `zarządzenie` jest przechowywane jako `legal_form`, a nie jako
poziom tej samej hierarchii co strategia/polityka/standard.

In [1]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT = Path(r"C:\Krzysiek\Pismak")
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from Zrodlowyszukiwator.rag_ingestion.catalog import load_catalog
from Zrodlowyszukiwator.rag_ingestion.embedder import OllamaEmbedder
from Zrodlowyszukiwator.rag_ingestion.pipeline import ingest_pdf
from Zrodlowyszukiwator.rag_ingestion.repository import PgVectorRepository
from Zrodlowyszukiwator.rag_ingestion.pdf_extractor import extract_pdf_pages
from Zrodlowyszukiwator.rag_ingestion.chunker import build_text_units, chunk_units

print("Gotowe.")

Gotowe.


In [2]:
# Hierarchia MERYTORYCZNA — nie jest to hierarchia prawna.
HIERARCHY = {
    "strategy": 100,
    "policy": 80,
    "program": 70,
    "plan": 70,
    "standard": 60,
    "procedure": 50,
    "other": 40,
}

HIERARCHY

{'strategy': 100,
 'policy': 80,
 'program': 70,
 'plan': 70,
 'standard': 60,
 'procedure': 50,
 'other': 40}

In [3]:
# Konfiguracja.
# Wskaż jeden plik dodawany w tej sesji.

POLICIES_DIR = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "RAW"
CATALOG_PATH = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "document_catalog.json"

PDF_PATH = POLICIES_DIR / "00_Strategia-rozwoju-Wroclawia-Wroclaw-2050.pdf"

print(CATALOG_PATH)
catalog = load_catalog(CATALOG_PATH)

if PDF_PATH.name not in catalog:
    raise KeyError(
        f"Brak metadanych dla {PDF_PATH.name!r} w {CATALOG_PATH}"
    )

metadata = catalog[PDF_PATH.name]
display(metadata)

C:\Krzysiek\Pismak\src\Zrodlowyszukiwator\RAG\document_catalog.json


DocumentMetadata(document_id='wroclaw-strategy-2050-2025', title='Strategia Rozwoju Wrocławia „Wrocław 2050”', document_kind=<DocumentKind.STRATEGY: 'strategy'>, city='Wrocław', issuer='Gmina Wrocław', adopting_body='Rada Miejska Wrocławia', legal_form=<LegalForm.RESOLUTION: 'resolution'>, adoption_act_number='XXIV/463/25', adoption_date=datetime.date(2025, 10, 20), valid_from=None, valid_to=None, is_active=True, source_url=None, language='pl', tags=['strategia', 'rozwój miasta', 'Wrocław 2050'], relations=[], notes='Dokument nadrzędny dla polityk, programów i planów działań w hierarchii merytorycznej RAG.')

In [4]:
# Etap A — ekstrakcja PDF i kontrola jakości tekstu.

pages = extract_pdf_pages(PDF_PATH)

print("Liczba stron:", len(pages))
print("Strony z bardzo małą ilością tekstu:",
      [p.page_number for p in pages if p.low_text_quality])

print("\n--- przykład strony ---\n")
print(pages[min(2, len(pages)-1)].text[:2500])

Liczba stron: 240
Strony z bardzo małą ilością tekstu: [1, 2, 13, 14, 16, 18, 20, 22, 66, 70, 75, 76, 91, 92, 96, 110, 126, 141, 142, 149, 150, 157, 166, 180, 182, 185, 186, 189, 190, 193, 194, 196, 198, 216, 238, 239, 240]

--- przykład strony ---

Strategia
Rozwoju Wrocławia
„Wrocław 2050”
Otwarci na przyszłość –
nastawieni na Wrocław
Dokument został przyjęty 20 października 2025 r.
przez Radę Miejską Wrocławia w formie uchwały nr XXIV/463/25.


In [5]:
# Etap B — sekcje i chunking.

units = build_text_units(pages)

chunks = chunk_units(
    units,
    metadata,
    max_chars=3500,
    min_chars=250,
    overlap_units=1,
)

print("Text units:", len(units))
print("Chunks:", len(chunks))

for chunk in chunks[:3]:
    print("\n" + "="*80)
    print("chunk_id:", chunk.chunk_id)
    print("pages:", chunk.page_start, "-", chunk.page_end)
    print("section:", " > ".join(chunk.section_path))
    print(chunk.text[:1200])

Text units: 648
Chunks: 431

chunk_id: wroclaw-strategy-2050-2025:00000:0a0ceab4f8b9
pages: 1 - 4
section: KOMITET STERUJĄCY DS. STRATEGII „WROCŁAW 2050” > Jakub Mazur – Wiceprezydent Wrocławia
„Wrocław 2050”

...

„Wrocław 2050”

nastawieni na Wrocław Dokument został przyjęty 20 października 2025 r. przez Radę Miejską Wrocławia w formie uchwały nr XXIV/463/25.

Łukasz Medeksza – zastępca dyrektora Departamentu Strategii i Zrównoważonego Rozwoju UMW

chunk_id: wroclaw-strategy-2050-2025:00001:ea3a2754bdd8
pages: 4 - 4
section: KOMITET STERUJĄCY DS. STRATEGII „WROCŁAW 2050” > Jarosław Delewski – dyrektor Departamentu Edukacji UMW
Łukasz Medeksza – zastępca dyrektora Departamentu Strategii i Zrównoważonego Rozwoju UMW

dr inż. arch. Jacek Barski – dyrektor Departamentu Urbanistyki i Architektury UMW

Jan Bujak – dyrektor Departamentu Nieruchomości i Eksploatacji UMW Lech Filipiak – były dyrektor Departamentu Nieruchomości i Eksploatacji UMW prof. dr hab. Andrzej Łoś – Biuro Strategii Mia

In [7]:
# Etap C — PostgreSQL + pgvector.

DATABASE_URL = os.environ["DATABASE_URL"]
EMBEDDING_DIM = int(os.getenv("RAG_EMBEDDING_DIM", "1024"))

repo = PgVectorRepository(
    DATABASE_URL,
    embedding_dim=EMBEDDING_DIM,
)
repo.ensure_schema()

print("Schema gotowy.")

Schema gotowy.


In [8]:
# Etap D — lokalne embeddingi przez Ollamę.

embedder = OllamaEmbedder(
    model=os.getenv("RAG_EMBEDDING_MODEL", "bge-m3"),
    base_url=os.getenv("OLLAMA_URL", "http://127.0.0.1:11434"),
    batch_size=16,
)

test_embedding = embedder.embed(["Test embeddingu dla dokumentów miejskich."])[0]
print("Wymiar embeddingu:", len(test_embedding))

if len(test_embedding) != EMBEDDING_DIM:
    raise ValueError(
        f"Model zwraca {len(test_embedding)} wymiarów, "
        f"a DB jest skonfigurowana na {EMBEDDING_DIM}."
    )

Wymiar embeddingu: 1024


In [9]:
# Etap E — pełny ingestion.
# Ponowne uruchomienie dla identycznego pliku nie tworzy duplikatów.

result = ingest_pdf(
    PDF_PATH,
    metadata,
    repository=repo,
    embedder=embedder,
    max_chars=3500,
    min_chars=250,
    overlap_units=1,
    force=False,
)

result

{'status': 'ingested',
 'document_id': 'wroclaw-strategy-2050-2025',
 'title': 'Strategia Rozwoju Wrocławia „Wrocław 2050”',
 'pages': 240,
 'chunks': 431,
 'low_text_pages': [1,
  2,
  13,
  14,
  16,
  18,
  20,
  22,
  66,
  70,
  75,
  76,
  91,
  92,
  96,
  110,
  126,
  141,
  142,
  149,
  150,
  157,
  166,
  180,
  182,
  185,
  186,
  189,
  190,
  193,
  194,
  196,
  198,
  216,
  238,
  239,
  240],
 'sha256': '01cf07b1d1046d92828124c5a3f4ba02f7adc3ec675b01c5bb3e9ee283c88417',
 'embedding_dim': 1024}

In [10]:
# Kontrola bazy.

doc_count, chunk_count = repo.counts()

print("Dokumenty w bazie:", doc_count)
print("Chunki w bazie:", chunk_count)

Dokumenty w bazie: 1
Chunki w bazie: 431


## Dodawanie następnego dokumentu

1. Skopiuj PDF do katalogu `polityki miejskie/`.
2. Dopisz jego metadata do `examples/document_catalog.json`.
3. Zmień `PDF_PATH`.
4. Uruchom komórki od ekstrakcji do ingestion.

Po ustabilizowaniu formatu metadanych notebook można zastąpić CLI:

```cmd
python -m Zrodlowyszukiwacz.rag_ingestion.cli ^
  --file "C:\Krzysiek\Pismak\polityki miejskie\nowy.pdf" ^
  --catalog examples\document_catalog.json
```